## Bước 7-8: Stratified K-Fold Setup & Model Training
### 7.1 Đọc dữ liệu và import thư viện

## Thiết lập khả năng tái lập

Seed được đặt đồng nhất cho Python, NumPy và PyTorch. Các script mới còn sử dụng generator có seed cho DataLoader.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import set_global_seed

SEED = 42
set_global_seed(SEED)
print(f"Đã thiết lập seed tái lập: {SEED}")


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score

df = pd.read_csv('../data/processed/depression_severity_preprocessed.csv')

print("Số dòng:", len(df))
print(df['label'].value_counts())

Số dòng: 3519
label
minimum     2555
moderate     393
mild         290
severe       281
Name: count, dtype: int64


### 7.2 Thiết lập Stratified 5-Fold


In [2]:
# Chuyển nhãn chữ thành số theo đúng thứ tự mức độ (quan trọng vì đây là bài toán ordinal)
label_map = {'minimum': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
df['label_num'] = df['label'].map(label_map)

X_text = df['text_classical'].values
y = df['label_num'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Kiểm tra thử: xem tỉ lệ nhãn ở mỗi fold có đồng đều không
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_text, y)):
    test_labels = y[test_idx]
    print(f"Fold {fold_idx+1}: train={len(train_idx)}, test={len(test_idx)}, "
          f"phân bố test={np.bincount(test_labels)}")

Fold 1: train=2815, test=704, phân bố test=[511  58  79  56]
Fold 2: train=2815, test=704, phân bố test=[511  58  79  56]
Fold 3: train=2815, test=704, phân bố test=[511  58  79  56]
Fold 4: train=2815, test=704, phân bố test=[511  58  78  57]
Fold 5: train=2816, test=703, phân bố test=[511  58  78  56]


### 7.3 Train Logistic Regression qua 5-Fold (đúng chuẩn chống leakage)

In [3]:
fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_text, y)):
    X_train_text, X_test_text = X_text[train_idx], X_text[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # QUAN TRỌNG: fit TF-IDF CHỈ trên tập train của fold này
    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.9)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)   # chỉ transform, không fit lại

    model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')

    fold_results.append({'fold': fold_idx+1, 'accuracy': acc, 'macro_f1': macro_f1, 'qwk': qwk})
    print(f"Fold {fold_idx+1}: Accuracy={acc:.4f}, Macro-F1={macro_f1:.4f}, QWK={qwk:.4f}")

results_df = pd.DataFrame(fold_results)
print("\n=== TRUNG BÌNH 5-FOLD ===")
print(results_df[['accuracy', 'macro_f1', 'qwk']].mean().round(4))

Fold 1: Accuracy=0.6747, Macro-F1=0.4501, QWK=0.3869
Fold 2: Accuracy=0.6207, Macro-F1=0.4028, QWK=0.2751
Fold 3: Accuracy=0.6562, Macro-F1=0.4181, QWK=0.3854
Fold 4: Accuracy=0.6648, Macro-F1=0.4372, QWK=0.3757
Fold 5: Accuracy=0.6501, Macro-F1=0.4454, QWK=0.3353

=== TRUNG BÌNH 5-FOLD ===
accuracy    0.6533
macro_f1    0.4307
qwk         0.3517
dtype: float64


### 7.4 Đóng gói hàm train/evaluate dùng chung

In [4]:
def run_kfold_classical(model_builder, X_text, y, skf, model_name="Model"):
    fold_results = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_text, y)):
        X_train_text, X_test_text = X_text[train_idx], X_text[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.9)
        X_train = vectorizer.fit_transform(X_train_text)
        X_test = vectorizer.transform(X_test_text)

        model = model_builder()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
        fold_results.append({'fold': fold_idx+1, 'accuracy': acc, 'macro_f1': macro_f1, 'qwk': qwk})

    results_df = pd.DataFrame(fold_results)
    means = results_df[['accuracy', 'macro_f1', 'qwk']].mean()
    print(f"=== {model_name} — Trung bình 5-Fold ===")
    print(means.round(4))
    return results_df, means

# Kiểm tra lại hàm này cho ra đúng kết quả như ô trước (LogReg)
logreg_results, logreg_means = run_kfold_classical(
    lambda: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    X_text, y, skf, model_name="Logistic Regression"
)

=== Logistic Regression — Trung bình 5-Fold ===
accuracy    0.6533
macro_f1    0.4307
qwk         0.3517
dtype: float64


### 7.5 Train SVM

In [5]:
from sklearn.svm import SVC

svm_results, svm_means = run_kfold_classical(
    lambda: SVC(kernel='linear', class_weight='balanced', random_state=42),
    X_text, y, skf, model_name="SVM"
)

=== SVM — Trung bình 5-Fold ===
accuracy    0.6547
macro_f1    0.4141
qwk         0.3235
dtype: float64


### 7.6 Train XGBoost (TF-IDF giảm chiều + đặc trưng thủ công)

### 7.6a Hàm đặc trưng thủ công (copy lại từ Bước 6)

In [6]:
import re

def extract_handcrafted_features(text):
    text_lower = text.lower()
    words = text_lower.split()
    n_words = len(words) if len(words) > 0 else 1

    features = {}
    features['word_count'] = len(words)

    first_person = ['i', 'me', 'my', 'mine', 'myself']
    features['first_person_ratio'] = sum(words.count(w) for w in first_person) / n_words

    negation = ['not', 'no', 'never', "n't", 'nothing', 'none']
    features['negation_ratio'] = sum(words.count(w) for w in negation) / n_words

    absolutist = ['always', 'never', 'everyone', 'nobody', 'everything', 'nothing',
                   'completely', 'totally', 'entirely']
    features['absolutist_ratio'] = sum(words.count(w) for w in absolutist) / n_words

    features['sentence_count'] = len(re.findall(r'[.!?]+', text))
    features['exclamation_ratio'] = text.count('!') / max(len(text), 1)

    warning_words = ['suicide', 'suicidal', 'kill', 'die', 'death', 'worthless', 'hopeless']
    features['warning_word_count'] = sum(text_lower.count(w) for w in warning_words)

    return features

handcrafted_all = df['text'].apply(extract_handcrafted_features).apply(pd.Series).values
print("Kích thước đặc trưng thủ công:", handcrafted_all.shape)

Kích thước đặc trưng thủ công: (3519, 7)


### 7.6b Train XGBoost qua 5-Fold

In [7]:
from xgboost import XGBClassifier
from sklearn.decomposition import TruncatedSVD

xgb_fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_text, y)):
    X_train_text, X_test_text = X_text[train_idx], X_text[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    hc_train, hc_test = handcrafted_all[train_idx], handcrafted_all[test_idx]

    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.9)
    X_train_tfidf = vectorizer.fit_transform(X_train_text)
    X_test_tfidf = vectorizer.transform(X_test_text)

    svd = TruncatedSVD(n_components=300, random_state=42)
    X_train_svd = svd.fit_transform(X_train_tfidf)
    X_test_svd = svd.transform(X_test_tfidf)

    X_train_final = np.hstack([X_train_svd, hc_train])
    X_test_final = np.hstack([X_test_svd, hc_test])

    model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='mlogloss'
    )
    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
    xgb_fold_results.append({'fold': fold_idx+1, 'accuracy': acc, 'macro_f1': macro_f1, 'qwk': qwk})
    print(f"Fold {fold_idx+1}: Accuracy={acc:.4f}, Macro-F1={macro_f1:.4f}, QWK={qwk:.4f}")

xgb_results_df = pd.DataFrame(xgb_fold_results)
xgb_means = xgb_results_df[['accuracy', 'macro_f1', 'qwk']].mean()
print("\n=== XGBoost — Trung bình 5-Fold ===")
print(xgb_means.round(4))

Fold 1: Accuracy=0.7457, Macro-F1=0.3243, QWK=0.1954
Fold 2: Accuracy=0.7372, Macro-F1=0.3090, QWK=0.1763
Fold 3: Accuracy=0.7372, Macro-F1=0.2781, QWK=0.1493
Fold 4: Accuracy=0.7401, Macro-F1=0.3079, QWK=0.1571
Fold 5: Accuracy=0.7326, Macro-F1=0.2934, QWK=0.1289

=== XGBoost — Trung bình 5-Fold ===
accuracy    0.7386
macro_f1    0.3026
qwk         0.1614
dtype: float64


### 7.6c XGBoost có xử lý imbalance (sample_weight)

In [8]:
from sklearn.utils.class_weight import compute_sample_weight

xgb_balanced_results = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_text, y)):
    X_train_text, X_test_text = X_text[train_idx], X_text[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    hc_train, hc_test = handcrafted_all[train_idx], handcrafted_all[test_idx]

    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.9)
    X_train_tfidf = vectorizer.fit_transform(X_train_text)
    X_test_tfidf = vectorizer.transform(X_test_text)

    svd = TruncatedSVD(n_components=300, random_state=42)
    X_train_svd = svd.fit_transform(X_train_tfidf)
    X_test_svd = svd.transform(X_test_tfidf)

    X_train_final = np.hstack([X_train_svd, hc_train])
    X_test_final = np.hstack([X_test_svd, hc_test])

    # Tính trọng số mẫu: lớp thiểu số được nhân trọng số cao hơn
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                            random_state=42, eval_metric='mlogloss')
    model.fit(X_train_final, y_train, sample_weight=sample_weights)
    y_pred = model.predict(X_test_final)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
    xgb_balanced_results.append({'fold': fold_idx+1, 'accuracy': acc, 'macro_f1': macro_f1, 'qwk': qwk})
    print(f"Fold {fold_idx+1}: Accuracy={acc:.4f}, Macro-F1={macro_f1:.4f}, QWK={qwk:.4f}")

xgb_bal_df = pd.DataFrame(xgb_balanced_results)
xgb_bal_means = xgb_bal_df[['accuracy', 'macro_f1', 'qwk']].mean()
print("\n=== XGBoost (balanced) — Trung bình 5-Fold ===")
print(xgb_bal_means.round(4))

Fold 1: Accuracy=0.7514, Macro-F1=0.3751, QWK=0.3107
Fold 2: Accuracy=0.7230, Macro-F1=0.3503, QWK=0.2235
Fold 3: Accuracy=0.7415, Macro-F1=0.3293, QWK=0.2659
Fold 4: Accuracy=0.7415, Macro-F1=0.3631, QWK=0.2640
Fold 5: Accuracy=0.7368, Macro-F1=0.3958, QWK=0.2778

=== XGBoost (balanced) — Trung bình 5-Fold ===
accuracy    0.7388
macro_f1    0.3627
qwk         0.2684
dtype: float64


## Bước 8 (tiếp): Train BiLSTM
### 8.1 Kiểm tra torch

In [9]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.13.0+cpu
CUDA available: False


### 8.2 Tokenize và xây vocabulary

In [10]:
from collections import Counter

X_neural_text = df['text_neural'].values

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    return text.split()

# Xây vocabulary từ TOÀN BỘ dữ liệu ở bước này chỉ để demo kích thước
# (khi train thật trong K-Fold, vocabulary sẽ được xây lại riêng trên từng phần train)
all_tokens = []
for text in X_neural_text:
    all_tokens.extend(simple_tokenize(text))

word_freq = Counter(all_tokens)
print("Tổng số từ (token):", len(all_tokens))
print("Số từ độc nhất (vocabulary size):", len(word_freq))
print("\n10 từ phổ biến nhất:", word_freq.most_common(10))

Tổng số từ (token): 302105
Số từ độc nhất (vocabulary size): 12703

10 từ phổ biến nhất: [('i', 14902), ('to', 10303), ('and', 9809), ('the', 7655), ('a', 6542), ('my', 5594), ('of', 4479), ('me', 3797), ('it', 3762), ('that', 3683)]


### 8.3 Xây dựng Vocabulary Encoder và PyTorch Dataset

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

MAX_LEN = 150  # dựa trên EDA Bước 4: trung vị ~80 từ, 75th percentile ~101 từ

class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.min_freq = min_freq

    def build(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(simple_tokenize(text))
        idx = 2
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.word2idx[word] = idx
                idx += 1

    def encode(self, text, max_len=MAX_LEN):
        tokens = simple_tokenize(text)
        ids = [self.word2idx.get(t, 1) for t in tokens]  # 1 = <UNK> nếu từ lạ
        ids = ids[:max_len]
        ids = ids + [0] * (max_len - len(ids))  # 0 = <PAD>, đệm cho đủ độ dài
        return ids

    def __len__(self):
        return len(self.word2idx)


class DepressionDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = self.vocab.encode(self.texts[idx], self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


print("Đã định nghĩa xong Vocabulary và Dataset. Kiểm tra nhanh:")
test_vocab = Vocabulary(min_freq=2)
test_vocab.build(X_neural_text[:100])
print("Vocab size (thử trên 100 dòng đầu):", len(test_vocab))
print("Ví dụ encode:", test_vocab.encode(X_neural_text[0])[:20])

Đã định nghĩa xong Vocabulary và Dataset. Kiểm tra nhanh:
Vocab size (thử trên 100 dòng đầu): 749
Ví dụ encode: [2, 3, 2, 4, 5, 6, 7, 8, 9, 1, 10, 11, 1, 12, 13, 14, 1, 15, 1, 16]


### 8.4 Kiến trúc model BiLSTM

In [12]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_classes=4, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 vì bidirectional

    def forward(self, x):
        embedded = self.embedding(x)                  # (batch, seq_len, embed_dim)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Ghép hidden state cuối cùng của chiều xuôi và chiều ngược
        final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(final_hidden)
        out = self.fc(out)
        return out

# Kiểm tra nhanh model chạy được không (dùng dummy data)
dummy_model = BiLSTMClassifier(vocab_size=1000)
dummy_input = torch.randint(0, 1000, (4, MAX_LEN))  # batch=4, seq_len=MAX_LEN
dummy_output = dummy_model(dummy_input)
print("Kích thước output:", dummy_output.shape)
print("(Kỳ vọng: (4, 4) — batch=4, num_classes=4)")

Kích thước output: torch.Size([4, 4])
(Kỳ vọng: (4, 4) — batch=4, num_classes=4)


### 8.5 Train BiLSTM qua 5-Fold

In [13]:
import torch.optim as optim
import time

def train_bilstm_fold(X_train_text, y_train, X_test_text, y_test, epochs=10, batch_size=32):
    # Xây vocab CHỈ từ train (chống leakage)
    vocab = Vocabulary(min_freq=2)
    vocab.build(X_train_text)

    train_dataset = DepressionDataset(X_train_text, y_train, vocab)
    test_dataset = DepressionDataset(X_test_text, y_test, vocab)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = BiLSTMClassifier(vocab_size=len(vocab))

    # Xử lý imbalance: tính trọng số lớp, đưa vào hàm loss
    class_weights = compute_sample_weight('balanced', y_train)
    class_weight_tensor = torch.tensor(
        [1.0 / max((y_train == c).sum(), 1) for c in range(4)], dtype=torch.float32
    )
    class_weight_tensor = class_weight_tensor / class_weight_tensor.sum() * 4

    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            outputs = model(batch_x)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.numpy())
            all_labels.extend(batch_y.numpy())

    return all_labels, all_preds


bilstm_fold_results = []
start_time = time.time()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_neural_text, y)):
    fold_start = time.time()
    X_train_f, X_test_f = X_neural_text[train_idx], X_neural_text[test_idx]
    y_train_f, y_test_f = y[train_idx], y[test_idx]

    labels_true, labels_pred = train_bilstm_fold(X_train_f, y_train_f, X_test_f, y_test_f, epochs=10)

    acc = accuracy_score(labels_true, labels_pred)
    macro_f1 = f1_score(labels_true, labels_pred, average='macro')
    qwk = cohen_kappa_score(labels_true, labels_pred, weights='quadratic')
    bilstm_fold_results.append({'fold': fold_idx+1, 'accuracy': acc, 'macro_f1': macro_f1, 'qwk': qwk})

    fold_time = time.time() - fold_start
    print(f"Fold {fold_idx+1}: Accuracy={acc:.4f}, Macro-F1={macro_f1:.4f}, QWK={qwk:.4f} (mất {fold_time:.1f}s)")

bilstm_results_df = pd.DataFrame(bilstm_fold_results)
bilstm_means = bilstm_results_df[['accuracy', 'macro_f1', 'qwk']].mean()
print(f"\nTổng thời gian: {time.time()-start_time:.1f}s")
print("=== BiLSTM — Trung bình 5-Fold ===")
print(bilstm_means.round(4))

Fold 1: Accuracy=0.6236, Macro-F1=0.3139, QWK=0.3066 (mất 90.7s)
Fold 2: Accuracy=0.5227, Macro-F1=0.3161, QWK=0.1479 (mất 92.1s)
Fold 3: Accuracy=0.6136, Macro-F1=0.3336, QWK=0.2336 (mất 90.5s)
Fold 4: Accuracy=0.5909, Macro-F1=0.3396, QWK=0.2525 (mất 87.7s)
Fold 5: Accuracy=0.4964, Macro-F1=0.2958, QWK=0.1836 (mất 85.8s)

Tổng thời gian: 446.8s
=== BiLSTM — Trung bình 5-Fold ===
accuracy    0.5695
macro_f1    0.3198
qwk         0.2248
dtype: float64


## Bước 8 (tiếp): Train DistilBERT
### 8.6 Kiểm tra thư viện transformers

In [14]:
import transformers
print("Transformers version:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
print("Import thành công")

c:\Users\Thanh Hue\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.14.1
Import thành công


### 8.7 Chia dữ liệu và tải DistilBERT

In [15]:
from sklearn.model_selection import train_test_split

# Chỉ 1 lần chia train/test (80/20), không dùng 5-fold để giảm thời gian train trên CPU
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_neural_text, y, test_size=0.2, stratify=y, random_state=42
)

print("Số mẫu train:", len(X_train_bert))
print("Số mẫu test:", len(X_test_bert))
print("Phân bố nhãn train:", np.bincount(y_train_bert))
print("Phân bố nhãn test:", np.bincount(y_test_bert))

# Tải tokenizer và model DistilBERT (lần đầu sẽ tải về máy, cần internet)
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

print("\nĐã tải xong tokenizer và model DistilBERT")

Số mẫu train: 2815
Số mẫu test: 704
Phân bố nhãn train: [2044  232  314  225]
Phân bố nhãn test: [511  58  79  56]


c:\Users\Thanh Hue\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thanh Hue\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3523.29it/s]
[transformers] DistilBertForSequenceClassif


Đã tải xong tokenizer và model DistilBERT


### 8.8 Tokenize dữ liệu

In [16]:
def tokenize_texts(texts, tokenizer, max_len=150):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

train_encodings = tokenize_texts(X_train_bert, tokenizer)
test_encodings = tokenize_texts(X_test_bert, tokenizer)

print("Kích thước input_ids (train):", train_encodings['input_ids'].shape)
print("Kích thước input_ids (test):", test_encodings['input_ids'].shape)
print("\nVí dụ tokenize câu đầu tiên:")
print(train_encodings['input_ids'][0][:20])

Kích thước input_ids (train): torch.Size([2815, 150])
Kích thước input_ids (test): torch.Size([704, 150])

Ví dụ tokenize câu đầu tiên:
tensor([  101,  1045,  2052,  2066,  2000,  4474,  2026,  2767,  2007,  1996,
        16056,  1997, 12358,  2011,  4306,  4804,  2014, 29525,  3021,  1012])


### 8.9 Tạo Dataset và DataLoader

In [17]:
class BertDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset_bert = BertDataset(train_encodings, y_train_bert)
test_dataset_bert = BertDataset(test_encodings, y_test_bert)

BATCH_SIZE = 8  # nhỏ để phù hợp máy CPU 16GB RAM

train_loader_bert = DataLoader(train_dataset_bert, batch_size=BATCH_SIZE, shuffle=True)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=BATCH_SIZE, shuffle=False)

print("Số batch train:", len(train_loader_bert))
print("Số batch test:", len(test_loader_bert))

Số batch train: 352
Số batch test: 88


### 8.10 Fine-tune DistilBERT

In [18]:
import time

device = torch.device('cpu')
model_bert.to(device)

# Xử lý imbalance
class_weight_bert = torch.tensor(
    [1.0 / max((y_train_bert == c).sum(), 1) for c in range(4)], dtype=torch.float32
)
class_weight_bert = class_weight_bert / class_weight_bert.sum() * 4
criterion_bert = nn.CrossEntropyLoss(weight=class_weight_bert.to(device))

optimizer_bert = optim.AdamW(model_bert.parameters(), lr=2e-5)

EPOCHS = 3
model_bert.train()
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0
    for batch_idx, batch in enumerate(train_loader_bert):
        optimizer_bert.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion_bert(outputs.logits, labels)
        loss.backward()
        optimizer_bert.step()

        epoch_loss += loss.item()

        # In tiến trình mỗi 50 batch để theo dõi
        if (batch_idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"  Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader_bert)}, "
                  f"Loss={loss.item():.4f}, Đã chạy: {elapsed:.0f}s")

    avg_loss = epoch_loss / len(train_loader_bert)
    print(f"=== Epoch {epoch+1}/{EPOCHS} hoàn thành, Loss trung bình: {avg_loss:.4f} ===\n")

total_time = time.time() - start_time
print(f"Tổng thời gian fine-tune: {total_time/60:.1f} phút")

  Epoch 1, Batch 50/352, Loss=1.4218, Đã chạy: 121s
  Epoch 1, Batch 100/352, Loss=1.3528, Đã chạy: 329s
  Epoch 1, Batch 150/352, Loss=1.2652, Đã chạy: 649s
  Epoch 1, Batch 200/352, Loss=1.2522, Đã chạy: 1342s
  Epoch 1, Batch 250/352, Loss=1.2089, Đã chạy: 1528s
  Epoch 1, Batch 300/352, Loss=0.3145, Đã chạy: 2540s
  Epoch 1, Batch 350/352, Loss=1.2522, Đã chạy: 3008s
=== Epoch 1/3 hoàn thành, Loss trung bình: 1.2672 ===

  Epoch 2, Batch 50/352, Loss=1.1289, Đã chạy: 3194s
  Epoch 2, Batch 100/352, Loss=1.2875, Đã chạy: 3323s
  Epoch 2, Batch 150/352, Loss=1.1603, Đã chạy: 3449s
  Epoch 2, Batch 200/352, Loss=1.2632, Đã chạy: 3652s
  Epoch 2, Batch 250/352, Loss=0.7226, Đã chạy: 3854s
  Epoch 2, Batch 300/352, Loss=1.0597, Đã chạy: 4056s
  Epoch 2, Batch 350/352, Loss=1.0530, Đã chạy: 4231s
=== Epoch 2/3 hoàn thành, Loss trung bình: 1.0355 ===

  Epoch 3, Batch 50/352, Loss=1.2202, Đã chạy: 6252s
  Epoch 3, Batch 100/352, Loss=0.6742, Đã chạy: 6366s
  Epoch 3, Batch 150/352, Loss=0

In [19]:
import os
os.makedirs('../models/distilbert_model', exist_ok=True)

model_bert.save_pretrained('../models/distilbert_model')
tokenizer.save_pretrained('../models/distilbert_model')
print("Đã lưu xong model vào models/distilbert_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.76it/s]

Đã lưu xong model vào models/distilbert_model


### 8.11 Đánh giá DistilBERT trên tập test

In [20]:
model_bert.eval()
all_preds_bert, all_labels_bert = [], []

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds_bert.extend(preds.cpu().numpy())
        all_labels_bert.extend(labels.cpu().numpy())

acc_bert = accuracy_score(all_labels_bert, all_preds_bert)
macro_f1_bert = f1_score(all_labels_bert, all_preds_bert, average='macro')
qwk_bert = cohen_kappa_score(all_labels_bert, all_preds_bert, weights='quadratic')

print("=== DistilBERT — Kết quả trên tập test ===")
print(f"Accuracy: {acc_bert:.4f}")
print(f"Macro-F1: {macro_f1_bert:.4f}")
print(f"QWK:      {qwk_bert:.4f}")

=== DistilBERT — Kết quả trên tập test ===
Accuracy: 0.7159
Macro-F1: 0.5518
QWK:      0.4956
